# H3 Toybox 🧸 (原作者:松音)

Minimax H3

**① 確認顯示卡**  
建議選 **L4 GPU**（要快可選 **G4 / A100**，但耗點較快；**請勿選 T4**）。更換請點右上角「連線」旁 **▾** →「變更執行階段類型」。

**② 啟動服務**  
點擊下方「一鍵啟動」區塊內的圓形播放鍵（首次需約 4～6 分鐘，請勿關閉分頁）。

**③ 開啟創作網頁**  
出現綠色完成卡片後，點下方大按鈕即可開始使用（首次請允許存取雲端硬碟）。

> <font color="#f59e0b"><b>⚠️ 不用時記得中斷連線，才不會一直扣點！</b></font>  
> <font color="#fcd34d">• <b>停止扣點</b>：點上方選單「執行階段」→「中斷連線並刪除執行階段」。</font>  
> <font color="#fcd34d">• <b>查詢剩餘點數</b>：點右上角「RAM／磁碟」→「查看資源」。</font>


In [ ]:
#@title 一鍵啟動
#@markdown 點左邊的圓形播放鍵開始啟動（首次需約 4～6 分鐘）；勾選 `SAVE_TO_DRIVE` 會自動將作品存到 Google 雲端硬碟（推薦）。
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
USE_DRIVE_CACHE = False
SAVE_API_KEYS = False
INCLUDE_REWRITER = True
INSTALL_IMAGE = True
TOYBOX_REPO = "https://github.com/dseditor/minimax-h3-colab.git"

import os, sys, json, time, shutil, subprocess, urllib.request
from pathlib import Path
from IPython.display import display, HTML, Javascript, update_display

LOG = Path('/tmp/h3_setup.log')
LOG.write_text(f'=== H3 Toybox setup {time.strftime("%Y-%m-%d %H:%M:%S")} ===\n', encoding='utf-8')
PROGRESS_ID = 'h3-toybox-v07-progress'
GPU_HINT = '請選 L4 GPU（推薦），要快可選 G4 / A100，但相對耗點也快。請勿選 T4。'
LAUNCH_ANCHOR = '🧸 打開 H3 Toybox ↗'

def _bar_html(step, title, detail='', kind='run'):
    colors = {'run': '#3b82f6', 'ok': '#16a34a', 'err': '#dc2626'}
    borders = {'run': '#93c5fd', 'ok': '#86efac', 'err': '#fca5a5'}
    bgs = {'run': '#eff6ff', 'ok': '#f0fdf4', 'err': '#fef2f2'}
    width = {1: 33, 2: 66, 3: 100}.get(step, 10)
    if kind == 'err':
        width = max(width, 33)
    color, border, bg = colors[kind], borders[kind], bgs[kind]
    detail_html = f'<div style="margin-top:8px;color:#555;font-size:13px;line-height:1.5;">{detail}</div>' if detail else ''
    return f'''
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Noto Sans TC',sans-serif;border:2px solid {border};
                border-radius:16px;padding:16px 20px;background:{bg};max-width:680px;box-sizing:border-box;">
      <div style="font-size:16px;font-weight:700;color:#111;line-height:1.4;">{title}</div>
      <div style="margin-top:10px;height:10px;background:#e5e7eb;border-radius:8px;overflow:hidden;">
        <div style="width:{width}%;height:100%;background:{color};transition:width .3s ease;"></div>
      </div>
      {detail_html}
    </div>
    '''

_progress_shown = False

def set_progress(step, title, detail='', kind='run'):
    global _progress_shown
    html = HTML(_bar_html(step, title, detail, kind))
    if not _progress_shown:
        display(html, display_id=PROGRESS_ID)
        _progress_shown = True
    else:
        update_display(html, display_id=PROGRESS_ID)

def log_write(text):
    with LOG.open('a', encoding='utf-8') as f:
        f.write(text if text.endswith('\n') else text + '\n')

def log_tail(n=30):
    if not LOG.exists():
        return '（沒有安裝紀錄）'
    lines = LOG.read_text(encoding='utf-8', errors='ignore').splitlines()
    return '\n'.join(lines[-n:]) if lines else '（沒有安裝紀錄）'

def fail(msg, extra_log=None):
    set_progress(3, '❌ 啟動沒有成功', msg, kind='err')
    print('❌ ' + msg)
    print('──── 最後幾行紀錄（方便排查）────')
    print(extra_log if extra_log is not None else log_tail(30))
    raise SystemExit

def stop(msg):
    set_progress(1, '❌ 還不能開始安裝', msg, kind='err')
    print('❌ ' + msg)
    raise SystemExit

def run_logged(cmd, fail_msg, check=True):
    with LOG.open('a', encoding='utf-8') as f:
        f.write('\n>>> ' + ' '.join(str(c) for c in cmd) + '\n')
        f.flush()
        r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
        f.write(f'[exit {r.returncode}]\n')
    if check and r.returncode != 0:
        fail(fail_msg)
    return r

def show_open_button():
    from google.colab import output
    proxy_url = None
    for js_expr in (
        'google.colab.kernel.proxyPort(8190, {cache: false})',
        'google.colab.kernel.proxyPort(8190)',
    ):
        try:
            res = output.eval_js(js_expr)
            if isinstance(res, str) and res.startswith('http'):
                proxy_url = res
                break
        except Exception:
            pass

    btn_css = """
<style>
  #h3-launch-wrap {
    font-family: -apple-system, BlinkMacSystemFont, "Noto Sans TC", sans-serif;
    max-width: 680px;
    margin: 12px 0 4px;
    padding: 18px 20px 20px;
    border-radius: 18px;
    background: #0f172a;
    box-sizing: border-box;
  }
  #h3-launch-wrap .h3-launch-hint {
    color: #e2e8f0;
    font-size: 14px;
    font-weight: 600;
    margin: 0 0 12px;
  }
  #h3-launch-wrap a.h3-launch-btn {
    display: inline-flex !important;
    align-items: center;
    justify-content: center;
    min-width: 280px;
    width: 100%;
    max-width: 640px;
    box-sizing: border-box;
    padding: 18px 28px !important;
    font-size: 20px !important;
    font-weight: 800 !important;
    color: #fff !important;
    text-decoration: none !important;
    border-radius: 14px !important;
    background: linear-gradient(135deg, #2563eb 0%, #db2777 100%) !important;
    box-shadow: 0 8px 24px rgba(37, 99, 235, 0.35);
    letter-spacing: 0.02em;
    line-height: 1.35;
    font-family: -apple-system, BlinkMacSystemFont, "Noto Sans TC", sans-serif !important;
    border: 0 !important;
  }
  #h3-launch-wrap a.h3-launch-btn:hover {
    filter: brightness(1.08);
    transform: translateY(-1px);
  }
</style>
"""
    if proxy_url:
        safe_url = proxy_url.replace('&', '&amp;').replace('"', '&quot;')
        display(HTML(btn_css + f"""
<div id="h3-launch-wrap">
  <p class="h3-launch-hint">啟動完成，請點下面的大按鈕開始創作</p>
  <a class="h3-launch-btn" href="{safe_url}" target="_blank" rel="noopener noreferrer">🧸 點我開啟創作網頁 ↗</a>
</div>
"""))
        log_write('open button rendered via proxyPort')
        return

    display(HTML(btn_css + """
<div id="h3-launch-wrap">
  <p class="h3-launch-hint">啟動完成，請點下面的連結開始創作</p>
</div>
"""))
    try:
        output.serve_kernel_port_as_window(8190, anchor_text='🧸 點我開啟創作網頁 ↗', skip_warning=True)
    except TypeError:
        output.serve_kernel_port_as_window(8190, anchor_text='🧸 點我開啟創作網頁 ↗')
    log_write('open button fallback to serve_kernel_port_as_window')

def inject_keepalive():
    try:
        display(Javascript(r"""
    (function(){
      if (window.__h3KeepAlive) {
        try { clearInterval(window.__h3KeepAlive.audioTimer); } catch (e) {}
        try { clearInterval(window.__h3KeepAlive.reconnectTimer); } catch (e) {}
      }
      try {
        if (!window.__h3KeepAliveAudio) {
          const Ctx = window.AudioContext || window.webkitAudioContext;
          const ctx = new Ctx();
          const osc = ctx.createOscillator();
          const g = ctx.createGain();
          g.gain.value = 0.00001;
          osc.connect(g);
          g.connect(ctx.destination);
          osc.start();
          window.__h3KeepAliveAudio = ctx;
        }
      } catch (e) {}
      const audioTimer = setInterval(function(){
        try {
          if (window.__h3KeepAliveAudio && window.__h3KeepAliveAudio.state === 'suspended') {
            window.__h3KeepAliveAudio.resume();
          }
          const doc = window.parent ? window.parent.document : document;
          doc.dispatchEvent(new MouseEvent('mousemove', {bubbles: true, clientX: 120, clientY: 120}));
          doc.dispatchEvent(new KeyboardEvent('keydown', {bubbles: true, key: 'Shift'}));
        } catch (e) {}
      }, 45000);
      const reconnectTimer = setInterval(function(){
        try {
          const docs = [document];
          try {
            if (window.parent && window.parent.document && window.parent.document !== document) {
              docs.push(window.parent.document);
            }
          } catch (e) {}
          for (const doc of docs) {
            const btn = doc.querySelector('colab-connect-button');
            if (btn && btn.shadowRoot) {
              const text = btn.shadowRoot.textContent || '';
              if (text.includes('Reconnect') || text.includes('重新連線') || text.includes('重新连接')) {
                btn.click();
              }
            }
          }
          fetch(window.location.href, {method: 'HEAD', cache: 'no-store'}).catch(function(){});
        } catch (e) {}
      }, 60000);
      window.__h3KeepAlive = {audioTimer: audioTimer, reconnectTimer: reconnectTimer};
    })();
    """))
        log_write('keepalive injected')
    except Exception as e:
        log_write(f'keepalive skipped: {e}')

set_progress(1, '步驟 1/3：檢查顯卡與準備環境中…', '請稍候，不要關閉這個分頁。')

# ---------- 多 GPU 偵測與防呆 ----------
import torch
gpu_ok = torch.cuda.is_available()
gpu_name, vram, cap, drv = '', 0.0, (0, 0), ''
if gpu_ok:
    cap = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        capture_output=True, text=True)
    log_write('nvidia-smi: ' + smi.stdout.strip() + '\n' + smi.stderr)
    drv = subprocess.run(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
        capture_output=True, text=True).stdout.strip()
    if not gpu_name and smi.returncode == 0 and smi.stdout.strip():
        gpu_name = smi.stdout.split(',')[0].strip()

if not gpu_ok:
    stop('目前沒有偵測到 GPU。請點右上角「連線」旁的 ▾ 小箭頭 → 變更執行階段類型 → ' + GPU_HINT + '選好後按儲存，再按一次播放鍵。')

if cap < (8, 0):
    if 'T4' in gpu_name:
        stop('目前偵測到 T4，顯存與架構跑不動這套模型。請點右上角「連線」旁小箭頭 → 變更執行階段類型 → ' + GPU_HINT + '選好後按儲存，再按一次播放鍵。')
    stop(f'目前偵測到 {gpu_name or "這張顯示卡"}（架構版本 {cap[0]}.{cap[1]}），顯存與架構跑不動這套模型。請點右上角「連線」旁小箭頭 → 變更執行階段類型 → {GPU_HINT}選好後按儲存，再按一次播放鍵。')

if not drv:
    stop('讀不到顯示卡驅動版本。請到「執行階段」→「中斷連線並刪除執行階段」，再重新連線後按一次播放鍵。')
try:
    drv_major = int(drv.split('.')[0])
except ValueError:
    drv_major = 0
if drv_major < 580:
    stop(f'目前的顯示卡驅動版本是 {drv}，需要 580 或以上才能跑。請到「執行階段」→「中斷連線並刪除執行階段」，再重新連線後按一次播放鍵。')

free_gib = round(shutil.disk_usage('/content').free / 2**30, 1)
_models_ready = Path('/content/ComfyUI/models/diffusion_models/Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors').is_file()
if shutil.disk_usage('/content').free < 72 * 2**30:
    if _models_ready:
        log_write(f'可用磁碟約 {free_gib} GiB（模型已下載過，略過 72 GiB 檢查，繼續啟動）')
    else:
        stop(f'磁碟空間不太夠（目前剩下約 {free_gib} GiB，需要至少 72 GiB），放不下大約 60 GB 的模型。請到「執行階段」→「中斷連線並刪除執行階段」，再重新連線後按一次播放鍵。')

hi = ('A100', 'H100', 'G4', 'RTX', '6000')
if any(k in gpu_name for k in hi):
    gpu_msg = f'✅ 偵測到顯示卡：{gpu_name}（約 {vram} GiB 顯存），屬於高階規格，跑起來會比 L4 更快'
elif 'L4' in gpu_name:
    gpu_msg = f'✅ 偵測到顯示卡：{gpu_name}（約 {vram} GiB 顯存），符合標準推薦規格'
else:
    gpu_msg = f'✅ 偵測到顯示卡：{gpu_name}（約 {vram} GiB 顯存），符合這套模型的需求'
log_write(gpu_msg)
log_write(f'驅動：{drv}｜可用磁碟：約 {free_gib} GiB')
set_progress(1, '步驟 1/3：檢查顯卡與準備環境中…', gpu_msg + '<br>正在準備環境，請稍候。')

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    log_write('已讀到 HF_TOKEN')
except Exception:
    log_write('沒有設定 HF_TOKEN，之後改用匿名下載（人多時可能被限速）')

def clone(url, dest):
    dest = Path(dest)
    if not dest.exists():
        run_logged(['git', 'clone', '--depth', '1', url, str(dest)],
                   f'下載套件失敗（{url}）。請確認網路連線後，再按一次播放鍵。')
    else:
        run_logged(['git', '-C', str(dest), 'pull', '--ff-only'],
                   f'更新套件失敗（{url}）', check=False)

COMFY = Path('/content/ComfyUI')
already_installed = (
    (COMFY / 'main.py').is_file()
    and Path('/content/minimax-h3-colab/toybox/server.py').is_file()
)

clone('https://github.com/Comfy-Org/ComfyUI.git', COMFY)

if already_installed:
    log_write('偵測到已經安裝過，略過重裝，改為快速更新。')
    log_write('skip pip: already installed')
else:
    if not (torch.version.cuda or '').startswith('13'):
        log_write('正在安裝相容的 PyTorch（第一次會多花一點時間）…')
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu130'],
                   '安裝 PyTorch 失敗。請看下方紀錄，或中斷連線後換一台執行階段再試。')
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt'),
                'huggingface_hub', 'hf_xet'],
               '安裝 ComfyUI 相依套件失敗。請看下方紀錄，再按一次播放鍵試試。')

# 工作流用到的三個節點組（都不會動到 PyTorch）
NODES = {
    'minimax-h3-audio-t8': 'https://github.com/T8mars/comfyui-minimax-h3-audio-T8',        # 語意橋 Semantic Bridge
    'comfyui-SelfLift': 'https://github.com/facok/comfyui-SelfLift',                      # 低解析度→潛空間放大→高解析度
    'ComfyUI-H3-Motion-Context': 'https://github.com/NikoDemon80/ComfyUI-H3-Motion-Context',  # 延伸影片（氣鎖）
}
for name, url in NODES.items():
    clone(url, COMFY / 'custom_nodes' / name)
# 導演 Skill 只是文字檔：放在 custom_nodes 外面，不載入它的節點
clone('https://github.com/T8mars/comfyui-minimax-h3-prompt-enhancer-T8', '/content/t8-enhancer')
# H3 Toybox 本體
clone(TOYBOX_REPO, '/content/minimax-h3-colab')
APP = Path('/content/minimax-h3-colab/toybox')
if shutil.which('ffmpeg'):
    log_write('ffmpeg: ' + str(shutil.which('ffmpeg')))
else:
    log_write('這台機器沒有 ffmpeg，之後若要「延伸影片」可能會失敗。')
pt = subprocess.run([sys.executable, '-c', 'import torch;print(torch.__version__, torch.version.cuda, torch.cuda.is_available())'],
                    capture_output=True, text=True)
log_write('PyTorch: ' + pt.stdout.strip() + ' ' + pt.stderr[-300:])

# ---------- 步驟 2：下載模型 ----------
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
try:
    from huggingface_hub.utils import disable_progress_bars
    disable_progress_bars()
except Exception:
    pass
from huggingface_hub import hf_hub_download

CACHE = None
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = Path('/content/drive/MyDrive/H3_models')
    CACHE.mkdir(parents=True, exist_ok=True)

ASSETS = 'dseditor/minimax-h3-studio-assets'
files = [  # (repo, 檔案, 放到 models/ 底下的哪裡)
    ('OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM', 'Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors', 'diffusion_models'),
    ('Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_int8_convrot.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
    # pika flex Ref2VA（2026-11-04 取代 dmad）：影片與 32×32 TTS 共用這一顆
    ('Comfy-Org/MiniMax-H3', 'loras/pika_h3_flex_ref2va_v1_bf16.safetensors', 'loras'),
] + [(ASSETS, f, str(Path(f).parent)) for f in [
    'loras/minimax_h3_turbo_v4_step600_ema_DasiwaREF2VAHybridV1_0_curveproj1025_compat_v001.safetensors',
    'loras/MysticXXX_MMH3-V2.safetensors',
    'loras/H3_Combat_V2.safetensors',
    'loras/Motion_Repair.safetensors',
    'loras/H3_SPEED_slider.safetensors',
    'semantic_bridge/t8_compat/BUNNY_H3_ActionLogic_Bridge_V1_T8_Compat.safetensors',
    'latent_upscale_models/minimax_h3_latent_upscaler_3d_fp16_V2.safetensors',
] + (['text_encoders/minimax_h3_rewriter_ref2va_qwen3.5_9b_w4a8.safetensors'] if INCLUDE_REWRITER else [])] + ([
    ('PrunaAI/P-H3-Image-Comfy', 'loras/p-h3-image_lora_bf16.safetensors', 'loras'),
    ('PrunaAI/P-H3-Image-Comfy', 'text_encoders/p-h3-image_text_encoder_int8_convrot.safetensors', 'text_encoders'),
    ('PrunaAI/P-H3-Image-Comfy', 'vae/p-h3-image_vae_int8_convrot.safetensors', 'vae'),
] if INSTALL_IMAGE else [])

t0 = time.time()
total = len(files)
set_progress(2, f'步驟 2/3：下載 AI 模型中（第 0/{total} 個檔案，首次約 4～6 分鐘，請勿關閉分頁）…',
             '已通過環境檢查，開始核對並下載模型。')
for i, (repo, name, folder) in enumerate(files, 1):
    target = COMFY / 'models' / folder / Path(name).name
    target.parent.mkdir(parents=True, exist_ok=True)
    cached = CACHE / Path(name).name if CACHE else None
    short = Path(name).name
    step2_title = f'步驟 2/3：下載 AI 模型中（第 {i}/{total} 個檔案，首次約 4～6 分鐘，請勿關閉分頁）…'
    try:
        if not target.is_file():
            if cached and cached.is_file():
                log_write(f'正在從雲端硬碟複製第 {i}/{total} 個檔案：{short}')
                set_progress(2, step2_title, f'正在從雲端硬碟複製第 {i}/{total} 個檔案')
                shutil.copy(str(cached), str(target))
            else:
                log_write(f'正在下載第 {i}/{total} 個檔案：{short}')
                set_progress(2, step2_title, f'正在下載第 {i}/{total} 個檔案')
                got = Path(hf_hub_download(repo_id=repo, filename=name, local_dir='/content/hf'))
                shutil.move(str(got), str(target))
        else:
            log_write(f'（{i}/{total}）已存在，略過：{short}')
            set_progress(2, step2_title, f'已核對第 {i}/{total} 個檔案')
        if cached and not cached.is_file():
            log_write(f'存一份到雲端硬碟：{cached.name}')
            shutil.copy(str(target), str(cached))
        log_write(f'{folder:<28} {target.name[:60]:<60} {target.stat().st_size / 2**30:5.1f} GiB')
    except Exception as e:
        fail(f'下載模型「{short}」時失敗：{e}。請確認網路或 HF_TOKEN 後再按一次播放鍵。')

# 官方提示詞寫作指南（改寫模型照它寫）
GUIDES = APP / 'guides'
GUIDES.mkdir(exist_ok=True)
for g in ('VIDEO_PROMPT_WRITING_GUIDE_ref_en.md', 'VIDEO_PROMPT_WRITING_GUIDE_base_en.md'):
    try:
        got = Path(hf_hub_download(repo_id='MiniMaxAI/MiniMax-H3', filename='docs/' + g, local_dir='/content/hf'))
        shutil.copy(str(got), str(GUIDES / g))
    except Exception as e:
        fail(f'下載提示詞寫作指南「{g}」時失敗：{e}。請確認網路後再按一次播放鍵。')
log_write(f'模型準備完成，{(time.time() - t0) / 60:.1f} 分鐘')

# ---------- 步驟 3：啟動服務 ----------
set_progress(3, '步驟 3/3：啟動 ComfyUI 與 Toybox 網頁中…', '模型已準備好，正在啟動服務。')

def wait_http(url, name, log, tries=180):
    for _ in range(tries):
        try:
            urllib.request.urlopen(url, timeout=2)
            log_write(name + ' 已啟動')
            return
        except Exception:
            time.sleep(2)
    text = ''
    try:
        text = open(log, encoding='utf-8', errors='ignore').read()[-3000:]
    except Exception:
        text = f'（讀不到 {log}）'
    fail(name + ' 沒有啟動成功。請看下方紀錄，或先用最底下的「系統診斷」查看。', extra_log=text)

for proc in ('server', 'toybox'):
    if proc in globals() and globals()[proc].poll() is None:
        globals()[proc].terminate()
        globals()[proc].wait()
# 同一個執行階段重跑時，清掉可能殘留、但不在變數裡的舊行程
subprocess.run(['pkill', '-f', '/content/ComfyUI/main.py'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['pkill', '-f', 'minimax-h3-colab/toybox/server.py'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 作品放哪裡：雲端硬碟（授權成功）或這台執行階段的暫存空間
DATA = Path('/content/h3toybox_data')
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA = Path('/content/drive/MyDrive/H3Toybox')
    except Exception as e:
        print('⚠️ 沒有掛上雲端硬碟（', e, '）：這次的作品只存在暫存空間，關掉執行階段就會消失')
        log_write(f'沒有掛上雲端硬碟（{e}）：改存暫存空間')
DATA.mkdir(parents=True, exist_ok=True)
# ComfyUI 的輸出資料夾直接設在 DATA/output。不能用連結把 output/H3Toybox 接過去：
# ComfyUI 存檔與 /view 都會解開連結檢查真實位置，指到 output 外面就報
# 「Saving image outside the output folder is not allowed」。
output_root = DATA / 'output'
videos, out_dir = output_root / 'H3Toybox', COMFY / 'output' / 'H3Toybox'
videos.mkdir(parents=True, exist_ok=True)
# 舊版留下的東西搬過來，以前生成的影片才看得到：DATA/videos、舊連結指向的資料夾、機器上的 output/H3Toybox
old_dirs = [DATA / 'videos']
if out_dir.is_symlink():
    old_dirs.append(out_dir.resolve())
    out_dir.unlink()
elif out_dir.is_dir():
    old_dirs.append(out_dir)
for old in old_dirs:
    if old.is_dir() and old.resolve() != videos.resolve():
        for f in old.iterdir():
            if f.is_file() and not (videos / f.name).exists():
                shutil.move(str(f), str(videos / f.name))

# ComfyUI 只當引擎：只聽 127.0.0.1，不對外開介面
server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
                           '--output-directory', str(output_root)],
                          cwd=str(COMFY), stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:8188/system_stats', 'ComfyUI', '/content/comfyui.log')
log_write('影片寫到：' + str(videos))

# H3 Toybox 的 Colab 設定（保留使用者在網頁設定頁改過的內容，只覆寫這台機器的路徑）
(COMFY / 'input').mkdir(exist_ok=True)
cfg_path = DATA / 'config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8')) if cfg_path.exists() else {}
cfg.update({
    'comfy_url': 'http://127.0.0.1:8188',
    'comfy_input_dir': str(COMFY / 'input'),
    'skills_dir': '/content/t8-enhancer/directional_skills',
    'free_after_rewrite': True,   # Colab 記憶體比較小：改寫完就把 9B 模型卸載
})
cfg.pop('guides', None)
# 圖片分頁：Colab 用 P-H3-Image（掛在 PRISM 上）；有沒有裝，看檔案是不是真的在
PH3 = ('loras/p-h3-image_lora_bf16.safetensors', 'text_encoders/p-h3-image_text_encoder_int8_convrot.safetensors',
       'vae/p-h3-image_vae_int8_convrot.safetensors')
# 短片 Agent：Colab 上沒有本地大模型，預設改走 Gemini（金鑰在網頁 ⚙ 設定填）
cfg['agent'] = dict(cfg.get('agent') or {}, provider=(cfg.get('agent') or {}).get('provider', 'gemini'))
cfg['image'] = dict(cfg.get('image') or {}, engine='ph3',
                    installed=all((COMFY / 'models' / f).is_file() for f in PH3))
log_write('生圖：' + ('已安裝' if cfg['image']['installed'] else '沒有安裝'))
cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding='utf-8')
secrets = DATA / 'secrets.json' if SAVE_API_KEYS else Path('/content/h3toybox_secrets.json')
toybox = subprocess.Popen([sys.executable, 'server.py'], cwd=str(APP),
                          env=dict(os.environ, H3TOYBOX_PORT='8190', H3TOYBOX_DATA=str(DATA), H3TOYBOX_SECRETS=str(secrets)),
                          stdout=open('/content/toybox.log', 'w'), stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:8190/api/config', 'H3 Toybox', '/content/toybox.log', tries=30)
log_write('作品存放位置：' + str(DATA))

set_progress(3, '✅ 啟動完成！請點擊下方大按鈕開始創作',
             f'{gpu_msg}<br>影片會存到：{videos}', kind='ok')
show_open_button()
inject_keepalive()


## 🔧 進階工具（平常不用展開）


In [ ]:
#@title 🔄 重啟服務
#@markdown 網頁打不開時再跑。不會重下載模型。
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
SAVE_API_KEYS = False  #@param {type:"boolean"}
import os, sys, json, time, shutil, subprocess, urllib.request
from pathlib import Path
from IPython.display import display, HTML, Javascript

COMFY = Path('/content/ComfyUI')
APP = Path('/content/minimax-h3-colab/toybox')
LAUNCH_ANCHOR = '🧸 打開 H3 Toybox ↗'
if not (COMFY / 'main.py').is_file() or not (APP / 'server.py').is_file():
    print('❌ 還沒安裝完成。請先跑上面的「▶ 一鍵啟動」，不要只跑這一格。')
    raise SystemExit

def wait_http(url, name, log, tries=180):
    for _ in range(tries):
        try:
            urllib.request.urlopen(url, timeout=2)
            print(name, '已啟動')
            return
        except Exception:
            time.sleep(2)
    try:
        print(open(log, encoding='utf-8', errors='ignore').read()[-3000:])
    except Exception:
        print('讀不到紀錄檔：', log)
    raise RuntimeError(name + ' 沒有啟動成功，看上面的紀錄')

def show_open_button():
    from google.colab import output
    proxy_url = None
    for js_expr in (
        'google.colab.kernel.proxyPort(8190, {cache: false})',
        'google.colab.kernel.proxyPort(8190)',
    ):
        try:
            res = output.eval_js(js_expr)
            if isinstance(res, str) and res.startswith('http'):
                proxy_url = res
                break
        except Exception:
            pass

    btn_css = """
<style>
  #h3-launch-wrap {
    font-family: -apple-system, BlinkMacSystemFont, "Noto Sans TC", sans-serif;
    max-width: 680px;
    margin: 12px 0 4px;
    padding: 18px 20px 20px;
    border-radius: 18px;
    background: #0f172a;
    box-sizing: border-box;
  }
  #h3-launch-wrap .h3-launch-hint {
    color: #e2e8f0;
    font-size: 14px;
    font-weight: 600;
    margin: 0 0 12px;
  }
  #h3-launch-wrap a.h3-launch-btn {
    display: inline-flex !important;
    align-items: center;
    justify-content: center;
    min-width: 280px;
    width: 100%;
    max-width: 640px;
    box-sizing: border-box;
    padding: 18px 28px !important;
    font-size: 20px !important;
    font-weight: 800 !important;
    color: #fff !important;
    text-decoration: none !important;
    border-radius: 14px !important;
    background: linear-gradient(135deg, #2563eb 0%, #db2777 100%) !important;
    box-shadow: 0 8px 24px rgba(37, 99, 235, 0.35);
    letter-spacing: 0.02em;
    line-height: 1.35;
    font-family: -apple-system, BlinkMacSystemFont, "Noto Sans TC", sans-serif !important;
    border: 0 !important;
  }
  #h3-launch-wrap a.h3-launch-btn:hover {
    filter: brightness(1.08);
    transform: translateY(-1px);
  }
</style>
"""
    if proxy_url:
        safe_url = proxy_url.replace('&', '&amp;').replace('"', '&quot;')
        display(HTML(btn_css + f"""
<div id="h3-launch-wrap">
  <p class="h3-launch-hint">啟動完成，請點下面的大按鈕開始創作</p>
  <a class="h3-launch-btn" href="{safe_url}" target="_blank" rel="noopener noreferrer">🧸 點我開啟創作網頁 ↗</a>
</div>
"""))
        return

    display(HTML(btn_css + """
<div id="h3-launch-wrap">
  <p class="h3-launch-hint">啟動完成，請點下面的連結開始創作</p>
</div>
"""))
    try:
        output.serve_kernel_port_as_window(8190, anchor_text='🧸 點我開啟創作網頁 ↗', skip_warning=True)
    except TypeError:
        output.serve_kernel_port_as_window(8190, anchor_text='🧸 點我開啟創作網頁 ↗')


for proc in ('server', 'toybox'):
    if proc in globals() and globals()[proc].poll() is None:
        globals()[proc].terminate()
        globals()[proc].wait()
subprocess.run(['pkill', '-f', '/content/ComfyUI/main.py'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['pkill', '-f', 'minimax-h3-colab/toybox/server.py'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 優先沿用一鍵啟動時的 DATA；沒有的話再依選項重建
if 'DATA' in globals() and Path(str(globals()['DATA'])).exists():
    DATA = Path(str(globals()['DATA']))
else:
    DATA = Path('/content/h3toybox_data')
    if SAVE_TO_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            DATA = Path('/content/drive/MyDrive/H3Toybox')
        except Exception as e:
            print('⚠️ 沒有掛上雲端硬碟（', e, '）：這次的作品只存在暫存空間，關掉執行階段就會消失')
DATA.mkdir(parents=True, exist_ok=True)
output_root = DATA / 'output'
videos, out_dir = output_root / 'H3Toybox', COMFY / 'output' / 'H3Toybox'
videos.mkdir(parents=True, exist_ok=True)
old_dirs = [DATA / 'videos']
if out_dir.is_symlink():
    old_dirs.append(out_dir.resolve())
    out_dir.unlink()
elif out_dir.is_dir():
    old_dirs.append(out_dir)
for old in old_dirs:
    if old.is_dir() and old.resolve() != videos.resolve():
        for f in old.iterdir():
            if f.is_file() and not (videos / f.name).exists():
                shutil.move(str(f), str(videos / f.name))

server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
                           '--output-directory', str(output_root)],
                          cwd=str(COMFY), stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:8188/system_stats', 'ComfyUI', '/content/comfyui.log')
print('🎬 影片寫到：', videos)

(COMFY / 'input').mkdir(exist_ok=True)
cfg_path = DATA / 'config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8')) if cfg_path.exists() else {}
cfg.update({
    'comfy_url': 'http://127.0.0.1:8188',
    'comfy_input_dir': str(COMFY / 'input'),
    'skills_dir': '/content/t8-enhancer/directional_skills',
    'free_after_rewrite': True,
})
cfg.pop('guides', None)
PH3 = ('loras/p-h3-image_lora_bf16.safetensors', 'text_encoders/p-h3-image_text_encoder_int8_convrot.safetensors',
       'vae/p-h3-image_vae_int8_convrot.safetensors')
cfg['agent'] = dict(cfg.get('agent') or {}, provider=(cfg.get('agent') or {}).get('provider', 'gemini'))
cfg['image'] = dict(cfg.get('image') or {}, engine='ph3',
                    installed=all((COMFY / 'models' / f).is_file() for f in PH3))
print('🖼 生圖：', '已安裝' if cfg['image']['installed'] else '沒有安裝（請先跑「一鍵啟動」）')
cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding='utf-8')
secrets = DATA / 'secrets.json' if SAVE_API_KEYS else Path('/content/h3toybox_secrets.json')
toybox = subprocess.Popen([sys.executable, 'server.py'], cwd=str(APP),
                          env=dict(os.environ, H3TOYBOX_PORT='8190', H3TOYBOX_DATA=str(DATA), H3TOYBOX_SECRETS=str(secrets)),
                          stdout=open('/content/toybox.log', 'w'), stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:8190/api/config', 'H3 Toybox', '/content/toybox.log', tries=30)
print('📁 作品存放位置：', DATA)

show_open_button()
try:
    display(Javascript(r"""
    (function(){
      if (window.__h3KeepAlive) {
        try { clearInterval(window.__h3KeepAlive.audioTimer); } catch (e) {}
        try { clearInterval(window.__h3KeepAlive.reconnectTimer); } catch (e) {}
      }
      try {
        if (!window.__h3KeepAliveAudio) {
          const Ctx = window.AudioContext || window.webkitAudioContext;
          const ctx = new Ctx();
          const osc = ctx.createOscillator();
          const g = ctx.createGain();
          g.gain.value = 0.00001;
          osc.connect(g);
          g.connect(ctx.destination);
          osc.start();
          window.__h3KeepAliveAudio = ctx;
        }
      } catch (e) {}
      const audioTimer = setInterval(function(){
        try {
          if (window.__h3KeepAliveAudio && window.__h3KeepAliveAudio.state === 'suspended') {
            window.__h3KeepAliveAudio.resume();
          }
          const doc = window.parent ? window.parent.document : document;
          doc.dispatchEvent(new MouseEvent('mousemove', {bubbles: true, clientX: 120, clientY: 120}));
          doc.dispatchEvent(new KeyboardEvent('keydown', {bubbles: true, key: 'Shift'}));
        } catch (e) {}
      }, 45000);
      const reconnectTimer = setInterval(function(){
        try {
          const docs = [document];
          try {
            if (window.parent && window.parent.document && window.parent.document !== document) {
              docs.push(window.parent.document);
            }
          } catch (e) {}
          for (const doc of docs) {
            const btn = doc.querySelector('colab-connect-button');
            if (btn && btn.shadowRoot) {
              const text = btn.shadowRoot.textContent || '';
              if (text.includes('Reconnect') || text.includes('重新連線') || text.includes('重新连接')) {
                btn.click();
              }
            }
          }
          fetch(window.location.href, {method: 'HEAD', cache: 'no-store'}).catch(function(){});
        } catch (e) {}
      }, 60000);
      window.__h3KeepAlive = {audioTimer: audioTimer, reconnectTimer: reconnectTimer};
    })();
    """))
    print('🛡️ 已開啟靜默防斷線保活')
except Exception as e:
    print('⚠️ 靜默保活沒有注入（', e, '）：不影響開網頁，只是沒有自動防斷線')


In [ ]:
#@title 備援公開網址
#@markdown 預設關閉。網址沒有密碼，不要公開分享。需要時再勾選並執行。
OPEN_CLOUDFLARE = False  #@param {type:"boolean"}
from pathlib import Path
if not OPEN_CLOUDFLARE:
    print('⏭ 已跳過（要用時勾選後再執行這格）')
else:
    import subprocess, re, time
    if not Path('/content/cloudflared').exists():
        subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
        subprocess.run(['chmod', '+x', '/content/cloudflared'], check=True)
    # 通道的輸出寫進檔案，不要接 PIPE：PIPE 沒人讀會塞滿，通道就卡住
    if 'tun' in globals() and tun.poll() is None:
        tun.terminate(); tun.wait()
    open('/content/cloudflared.log', 'w').close()
    tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8190'],
                           stdout=open('/content/cloudflared.log', 'a'), stderr=subprocess.STDOUT)
    url = None
    for i in range(60):
        time.sleep(1)
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
        if m:
            url = m.group(0); break
    print('👉 H3 Toybox：', url or '（沒拿到網址，看 /content/cloudflared.log）')


In [ ]:
#@title 🔍 系統診斷
import subprocess
from pathlib import Path
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)

def show_log(path, limit=4000, keys=None, title=None):
    print(f'---- {title or path} ----')
    p = Path(path)
    if not p.is_file():
        print(f'（還沒有 {path}）')
        return
    text = p.read_text(encoding='utf-8', errors='ignore')
    if keys:
        filtered = '\n'.join(l for l in text.splitlines() if any(k in l for k in keys))
        print(filtered[-limit:] if filtered else '（沒有找到錯誤或關鍵字）')
    else:
        print(text[-limit:])

keys = ('error', 'Error', 'Traceback', 'out of memory', 'SelfLift timing] high_resolution total', 'Prompt executed', 'IMPORT FAILED')
show_log('/content/comfyui.log', 4000, keys, 'ComfyUI（篩過的關鍵行）')
show_log('/content/toybox.log', 2000, None, 'H3 Toybox')
show_log('/tmp/h3_setup.log', 3000, None, '安裝紀錄 /tmp/h3_setup.log')
